In [10]:
import os
import sys
import pandas as pd
import  random
import shutil

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling  import RandomOverSampler



# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config


In [2]:
#  Imports & Configuration

# User parameters

if config.USE_SAMPLED_TRAIN_DATASET:
    INPUT_DIR    = config.SAMPLED_TRAIN_DATASET_DIR         # path of sampled training dataset
else:
    INPUT_DIR    = config.TRAIN_DATASET_DIR                  # path of training dataset

PREPROCESSED_OUTPUT_DIR   = config.PREPROCESSED_DATASET_DIR
TARGET_SIZE  = (640, 640)                          # (height, width)
EXTS         = [".jpg", ".png", ".tif", ".tiff"]   # supported extensions

# CLAHE & denoise settings
CLAHE_CFG    = {"clip_limit": 2.0, "grid_size": (8, 8)}
DENOISE_CFG  = {"h": 10, "template_size": 7, "search_size": 21}

# Ensure output directory exists
os.makedirs(PREPROCESSED_OUTPUT_DIR, exist_ok=True)


In [11]:
from imblearn.over_sampling import RandomOverSampler

data_root    = config.TRAIN_DATASET_HPC_DIR
labels_csv   = config.TRAIN_LABELS_PATH 

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

box_size    = 32

total_cap     = 2000
neg_cap       = total_cap // 2   # 1000 negatives
pos_cap       = total_cap - neg_cap  # 1000 positives

pipeline = Pipeline([
    ("undersample", RandomUnderSampler(sampling_strategy={0: neg_cap}, random_state=42)),
    ("oversample",  RandomOverSampler (sampling_strategy={1: pos_cap}, random_state=42)),
])

for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id    = row["tomo_id"]
    z          = int(row["Motor axis 0"])
    y          = row["Motor axis 1"]
    x          = row["Motor axis 2"]
    img_width  = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name   = f"slice_{z:04d}.jpg"
    img_path   = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)


# 1) Build a DataFrame of all image paths + binary label
all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

df = pd.DataFrame({
    "path": all_images,
    "label": [1 if p in positive_samples else 0 for p in all_images]
})

print(df)

# 2) Oversample so positives == negatives (1:1 ratio)
X_res, y_res = pipeline.fit_resample(df[["path"]], df["label"])
balanced_paths = X_res["path"].tolist()
balanced_images = X_res["path"].tolist()

print(X_res)
print(y_res)


# 3) Shuffle & split into train/val
random.seed(42)
random.shuffle(balanced_images)
split_idx    = int(len(balanced_images) * 0.8)
train_images = balanced_images[:split_idx]
val_images   = balanced_images[split_idx:]

# 4) Process as before
def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        if not os.path.isfile(img_path):
            print(f"⚠️  Skipping missing image: {img_path}")
            continue
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z       = int(os.path.splitext(os.path.basename(img_path))[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        dst_img = os.path.join(img_out_dir,  f"{base_fn}.jpg")
        dst_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")
        shutil.copy(img_path, dst_img)

        lines = []
        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            cx, cy = x/w, y/h
            bw, bh = (box_size*2)/w, (box_size*2)/h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        with open(dst_lbl, "w") as f:
            f.write("\n".join(lines))

# finally call
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images,   output_images_val,   output_labels_val)


                                                     path  label
0       /data/horse/ws/kein254g-team_project/train/tom...      0
1       /data/horse/ws/kein254g-team_project/train/tom...      0
2       /data/horse/ws/kein254g-team_project/train/tom...      0
3       /data/horse/ws/kein254g-team_project/train/tom...      0
4       /data/horse/ws/kein254g-team_project/train/tom...      0
...                                                   ...    ...
269189  /data/horse/ws/kein254g-team_project/train/tom...      0
269190  /data/horse/ws/kein254g-team_project/train/tom...      0
269191  /data/horse/ws/kein254g-team_project/train/tom...      0
269192  /data/horse/ws/kein254g-team_project/train/tom...      0
269193  /data/horse/ws/kein254g-team_project/train/tom...      0

[269194 rows x 2 columns]
                                                   path
0     /data/horse/ws/kein254g-team_project/train/tom...
1     /data/horse/ws/kein254g-team_project/train/tom...
2     /data/horse/ws/kein

/home/kein254g/.local/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/kein254g/.local/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
/home/kein254g/.local/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/kein254g/.local/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [2]:
# Creates dataset to fit yolo format 
data_root    = config.TRAIN_DATASET_HPC_DIR
labels_csv   = config.TRAIN_LABELS_PATH 

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

box_size    = 32


for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id    = row["tomo_id"]
    z          = int(row["Motor axis 0"])
    y          = row["Motor axis 1"]
    x          = row["Motor axis 2"]
    img_width  = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name   = f"slice_{z:04d}.jpg"
    img_path   = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)

all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

# 1) Configuration for balancing
neg_pos_ratio = 1  # number of negative samples per positive

# 2) Separate positive & negative image paths
pos_images = list(positive_samples.keys())
neg_images = [img for img in all_images if img not in positive_samples]

# 3) Undersample negatives to at most neg_pos_ratio * #positives
num_pos = len(pos_images)
num_neg_keep = min(len(neg_images), num_pos * neg_pos_ratio)
neg_keep = random.sample(neg_images, num_neg_keep)

# 4) Combine & shuffle
balanced_images = pos_images + neg_keep
random.shuffle(balanced_images)

# 5) Train/val split on the balanced set
split_idx    = int(len(balanced_images) * 0.8)
train_images = balanced_images[:split_idx]
val_images   = balanced_images[split_idx:]

def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        # 1) Skip if the source image file doesn't exist
        if not os.path.isfile(img_path):
            print(f"⚠️  Skipping missing image: {img_path}")
            continue

        # 2) Reconstruct tomo_id and slice index as before
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z       = int(os.path.splitext(os.path.basename(img_path))[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        
        dst_img = os.path.join(img_out_dir,  f"{base_fn}.jpg")
        dst_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")

        # 3) Copy the image
        shutil.copy(img_path, dst_img)

        # 4) Write the label file (empty for negatives)
        lines = []
        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            cx = x / w
            cy = y / h
            bw = box_size * 2 / w
            bh = box_size * 2 / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        with open(dst_lbl, "w") as f:
            f.write("\n".join(lines))


# Process both sets
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images,   output_images_val,   output_labels_val)

print(f"Dataset created with {len(train_images)} training and {len(val_images)} validation images.") 


Dataset created with 720 training and 180 validation images.


Convert raw sample data to yolo format

[Object Detection dataset overview](https://docs.ultralytics.com/datasets/detect/)

Data augmentation for training data using albumentations liblary.

In [7]:


# 1) Set paths – adjust these!
root_dir    = config.TRAIN_DATASET_DIR           # parent folder containing tomo_* subfolders
labels_csv  = config.TRAIN_LABELS_PATH      # original CSV with a 'tomo_id' column
output_csv  = config.FILTERED_TRAIN_LABELS  # where to write the filtered CSV

print(os.listdir(root_dir))
# 2) Detect all tomo_ids by listing subfolders
tomo_ids = [
    name for name in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, name))
]
print(f"Detected {len(tomo_ids)} tomo_id folders.")
print(tomo_ids)

# 3) Load your full labels CSV
df = pd.read_csv(labels_csv)

# 4) Filter rows where 'tomo_id' is in the detected folders
filtered_df = df[df['tomo_id'].isin(tomo_ids)]
print(f"Filtering yields {len(filtered_df)} rows out of {len(df)} total.")

# 5) Write out the new CSV
filtered_df.to_csv(output_csv, index=False)
print(f"Written filtered CSV to {output_csv}")

# 6) (Optional) Preview first few rows
print(filtered_df.head())


['tomo_3183d2', 'tomo_1af88d', 'tomo_0363f2', 'tomo_1446aa', 'tomo_8d231b', 'tomo_04d42b', 'tomo_6df2d6', 'tomo_56b9a3', 'tomo_3e7783', 'tomo_2483bb', 'tomo_0a8f05', 'tomo_49725c', 'tomo_1e9980', 'tomo_935ae0', 'tomo_79756f', 'tomo_30b580', 'tomo_072a16', 'tomo_6bc974', 'tomo_79a385', 'tomo_4469a7', 'tomo_9c0253', 'tomo_3c6038', 'tomo_88af60', 'tomo_62eea8', '.DS_Store', 'tomo_2e1f4c']
Detected 25 tomo_id folders.
['tomo_3183d2', 'tomo_1af88d', 'tomo_0363f2', 'tomo_1446aa', 'tomo_8d231b', 'tomo_04d42b', 'tomo_6df2d6', 'tomo_56b9a3', 'tomo_3e7783', 'tomo_2483bb', 'tomo_0a8f05', 'tomo_49725c', 'tomo_1e9980', 'tomo_935ae0', 'tomo_79756f', 'tomo_30b580', 'tomo_072a16', 'tomo_6bc974', 'tomo_79a385', 'tomo_4469a7', 'tomo_9c0253', 'tomo_3c6038', 'tomo_88af60', 'tomo_62eea8', 'tomo_2e1f4c']
Filtering yields 26 rows out of 737 total.
Written filtered CSV to /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/filtered_train_labels.csv
    row_id      tomo_id  Motor axis 0  Mo

In [11]:
from sklearn.model_selection import train_test_split

# 1) Load your full slice‐level DataFrame
#    Assume df has one row per slice with columns ['tomo_id','slice','Number of motors','Array shape (axis 0)']
df = pd.read_csv(config.FILTERED_TRAIN_LABELS)

# 2) Build a slice‐level table with a binary label
df_slices = (
    df
    .drop_duplicates(subset=['tomo_id','Motor axis 2'])  # one slice per motor‐axis index
    .rename(columns={'Motor axis 2':'slice','Number of motors':'n_motors'})
)
df_slices['pos'] = df_slices['n_motors'] > 0

# 3) Split into positive and negative slices
pos_df = df_slices[df_slices['pos']]
neg_df = df_slices[~df_slices['pos']]

# 4) Undersample negatives to, say, 2× the positives
neg_sampled = neg_df.sample(
    n = min(len(neg_df), len(pos_df) * 2),
    random_state=42
)

# 5) (Optionally) Oversample positives up to match negatives
#    e.g. double the positives
pos_oversampled = pos_df.sample(
    n = min(len(pos_df)*2, len(neg_sampled)),
    replace=True,
    random_state=42
)

# 6) Stitch together and shuffle
balanced = pd.concat([neg_sampled, pos_oversampled]).sample(frac=1, random_state=42)

# 7) Now do your train/val split on the slice‐level IDs
train_slices, val_slices = train_test_split(
    balanced,
    test_size=0.2,
    stratify=balanced['pos'],
    random_state=42
)

# 8) Extract the lists you feed into your YOLO writer
train_images = train_slices.apply(lambda r: f"{r.tomo_id}_{r.slice}", axis=1).tolist()
val_images   = val_slices.apply  (lambda r: f"{r.tomo_id}_{r.slice}", axis=1).tolist()

print(f"Train: {len(train_images)} slices ({train_slices['pos'].mean()*100:.1f}% positive)")
print(f" Val : {len(val_images)} slices ({val_slices  ['pos'].mean()*100:.1f}% positive)")

print(train_images)
print(val_images)


Train: 9 slices (55.6% positive)
 Val : 3 slices (33.3% positive)
['tomo_04d42b_-1.0', 'tomo_4469a7_-1.0', 'tomo_30b580_575.0', 'tomo_3183d2_475.0', 'tomo_8d231b_-1.0', 'tomo_30b580_575.0', 'tomo_6bc974_610.0', 'tomo_1e9980_-1.0', 'tomo_49725c_739.0']
['tomo_9c0253_821.0', 'tomo_072a16_-1.0', 'tomo_88af60_-1.0']


In [14]:
from sklearn.model_selection import train_test_split

# -------- CONFIGURE PATHS & HYPERPARAMETERS --------
csv_path    = config.FILTERED_TRAIN_LABELS   # your annotation CSV
volumes_dir = config.TRAIN_DATASET_DIR               # e.g. data/volumes/<tomo_id>/slice_0001.png
out_dir     =  config.YOLO_DATA_DIR         # will contain train/ & val/
train_ratio = 0.8
neg_pos_ratio = 1                           # #negatives per positive
box_size    = 32                            # square box side in px
img_ext     = '.jpg'                        # or .jpg

# -------- 1) Build slice-level table --------
df = pd.read_csv(csv_path)
slices = []
for tomo_id, g in df.groupby('tomo_id'):
    # how many slices in this volume?
    total_slices = int(g['Array shape (axis 2)'].iloc[0])
    # for each z-slice, count motors
    for z in range(total_slices):
        nm = int((g['Motor axis 2'] == z).sum())
        slices.append({'tomo_id': tomo_id, 'slice': z, 'num_motors': nm})
slice_df = pd.DataFrame(slices)
slice_df['is_pos'] = slice_df['num_motors'] > 0

# -------- 2) Balance pos/neg --------
pos_df = slice_df[slice_df['is_pos']]
neg_df = slice_df[~slice_df['is_pos']]

# undersample negatives to neg_pos_ratio * #positives
neg_keep = neg_df.sample(n=min(len(neg_df), len(pos_df)*neg_pos_ratio),
                         random_state=42)
# optionally oversample positives to match neg_keep
pos_keep = pos_df.sample(n=len(pos_df), replace=True, random_state=42)

balanced = pd.concat([pos_keep, neg_keep]).sample(frac=1, random_state=42)

# -------- 3) Train/Val split --------
train_df, val_df = train_test_split(
    balanced,
    test_size=1-train_ratio,
    stratify=balanced['is_pos'],
    random_state=42
)

# -------- 4) Prepare YOLO folders --------
for split in ['train','val']:
    for sub in ['images','labels']:
        d = os.path.join(out_dir, split, sub)
        os.makedirs(d, exist_ok=True)

# helper to write one slice
def write_slice(row, split):
    img_name = f"{row.tomo_id}_slice_{row.slice:04d}{img_ext}"
    src_img  = os.path.join(volumes_dir, row.tomo_id, img_name)
    dst_img  = os.path.join(out_dir, split, 'images', img_name)
    # copy image
    shutil.copy(src_img, dst_img)

    # build label file
    lbl_path = os.path.join(out_dir, split, 'labels', img_name.replace(img_ext,'.txt'))
    lines = []
    if row.num_motors > 0:
        motors = df[
            (df.tomo_id==row.tomo_id) &
            (df['Motor axis 2']==row.slice)
        ]
        # you need image width/height to normalize; here we assume square imgs
        # load one image to get size:
        from PIL import Image
        w,h = Image.open(src_img).size
        for _, m in motors.iterrows():
            x_c = m['Motor axis 0']
            y_c = m['Motor axis 1']
            # center-normalized
            x = x_c / w
            y = y_c / h
            # box normalized
            bw = box_size / w
            bh = box_size / h
            lines.append(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, 'w') as f:
        f.write('\n'.join(lines))

# write out
for _, row in train_df.iterrows():
    write_slice(row, 'train')
for _, row in val_df.iterrows():
    write_slice(row, 'val')

print(f"► Done! Train: {len(train_df)} slices, Val: {len(val_df)} slices")



FileNotFoundError: [Errno 2] No such file or directory: '/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_8d231b/tomo_8d231b_slice_0874.jpg'

In [15]:
# Updated slice-writing code with file-existence checks

# Configuration (adjust as needed)
csv_path    = config.FILTERED_TRAIN_LABELS   
volumes_dir = config.TRAIN_DATASET_DIR              
out_dir     =  config.YOLO_DATA_DIR 
img_ext     = '.jpg'
box_size    = 32

# Load annotation CSV
df = pd.read_csv(csv_path)

# Build slice-level DataFrame
slices = []
for tomo_id, g in df.groupby('tomo_id'):
    total_slices = int(g['Array shape (axis 2)'].iloc[0])
    for z in range(total_slices):
        nm = int((g['Motor axis 2'] == z).sum())
        slices.append({'tomo_id': tomo_id, 'slice': z, 'num_motors': nm})
slice_df = pd.DataFrame(slices)

# Balance & split (example 80/20, 2:1 neg:pos)
pos_df = slice_df[slice_df['num_motors'] > 0]
neg_df = slice_df[slice_df['num_motors'] == 0]
neg_keep = neg_df.sample(n=min(len(neg_df), len(pos_df)*2), random_state=42)
pos_keep = pos_df.sample(n=len(pos_df), replace=True, random_state=42)
balanced = pd.concat([pos_keep, neg_keep]).sample(frac=1, random_state=42)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    balanced, test_size=0.2, stratify=balanced['num_motors']>0, random_state=42
)

# Prepare YOLO folders
for split in ['train','val']:
    for sub in ['images','labels']:
        os.makedirs(os.path.join(out_dir, split, sub), exist_ok=True)

# Function to write one slice, skipping missing files
def write_slice(row, split):
    img_name = f"{row.tomo_id}_slice_{row.slice:04d}{img_ext}"
    src_img  = os.path.join(volumes_dir, row.tomo_id, img_name)
    dst_img  = os.path.join(out_dir, split, 'images', img_name)
    lbl_path = os.path.join(out_dir, split, 'labels', img_name.replace(img_ext,'.txt'))

    print(src_img + "  "  + dst_img)


    # Skip if source image doesn't exist
    if not os.path.isfile(src_img):
        print(f"⚠️ Missing image, skipping: {src_img}")
        return

    # Copy image
    shutil.copy(src_img, dst_img)

    # Write label file
    lines = []
    if row.num_motors > 0:
        motors = df[(df.tomo_id==row.tomo_id) & (df['Motor axis 2']==row.slice)]
        w, h = Image.open(src_img).size
        for _, m in motors.iterrows():
            x_c, y_c = m['Motor axis 0'], m['Motor axis 1']
            x = x_c / w
            y = y_c / h
            bw = box_size / w
            bh = box_size / h
            lines.append(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, 'w') as f:
        f.write('\n'.join(lines))

# Write out train and val slices
for _, row in train_df.iterrows():
    write_slice(row, 'train')
for _, row in val_df.iterrows():
    write_slice(row, 'val')

print("✅ Preprocessing complete. Slices written with missing files skipped.")


/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_49725c/tomo_49725c_slice_0166.jpg  /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/yolo/train/images/tomo_49725c_slice_0166.jpg
⚠️ Missing image, skipping: /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_49725c/tomo_49725c_slice_0166.jpg
/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_3183d2/tomo_3183d2_slice_0475.jpg  /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/yolo/train/images/tomo_3183d2_slice_0475.jpg
⚠️ Missing image, skipping: /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_3183d2/tomo_3183d2_slice_0475.jpg
/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train/tomo_0a8f05/tomo_0a8f05_slice_0214.jpg  /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/yolo/train/images/tomo_0a8f05_slice_0214.jpg
⚠️ Missing image, skipping: /home/h4/

In [18]:


# Configuration (adjust as needed)
csv_path    = config.TRAIN_LABELS_PATH   
volumes_dir = config.TRAIN_DATASET_DIR              
out_dir     =  config.YOLO_DATA_DIR 
img_ext     = '.jpg'
box_size    = 32

# Load annotation CSV
df = pd.read_csv(csv_path)

# Build slice-level DataFrame
slices = []
for tomo_id, g in df.groupby('tomo_id'):
    total_slices = int(g['Array shape (axis 2)'].iloc[0])
    for z in range(total_slices):
        nm = int((g['Motor axis 2'] == z).sum())
        slices.append({'tomo_id': tomo_id, 'slice': z, 'num_motors': nm})
slice_df = pd.DataFrame(slices)

# Example balancing and train/val split (adjust to your pipeline)
# ... (assume train_df and val_df are defined here)

# Prepare YOLO folders
for split in ['train','val']:
    for sub in ['images','labels']:
        os.makedirs(os.path.join(out_dir, split, sub), exist_ok=True)

# Function to write one slice, handling filename layout
def write_slice(row, split):
    # source filename is just slice_{:04d}.jpg inside tomo folder
    src_img_name = f"slice_{row.slice:04d}{img_ext}"
    src_img_path = os.path.join(volumes_dir, row.tomo_id, src_img_name)
    
    # destination uses combined naming for uniqueness
    dst_img_name = f"{row.tomo_id}_slice_{row.slice:04d}{img_ext}"
    dst_img_path = os.path.join(out_dir, split, 'images', dst_img_name)
    
    # label file path
    lbl_path = os.path.join(out_dir, split, 'labels', dst_img_name.replace(img_ext, '.txt'))
    
    print(src_img_name + "  "  + src_img_path)

    # Skip missing files
    if not os.path.isfile(src_img_path):
        print(f"⚠️ Missing image, skipping: {src_img_path}")
        return

    # Copy image
    shutil.copy(src_img_path, dst_img_path)

    # Write label file
    lines = []
    if row.num_motors > 0:
        motors = df[(df.tomo_id == row.tomo_id) & (df['Motor axis 2'] == row.slice)]
        w, h = Image.open(src_img_path).size
        for _, m in motors.iterrows():
            x_c, y_c = m['Motor axis 0'], m['Motor axis 1']
            x = x_c / w
            y = y_c / h
            bw = box_size / w
            bh = box_size / h
            lines.append(f"0 {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, 'w') as f:
        f.write('\n'.join(lines))

# Example usage:
# for _, row in train_df.iterrows():
#     write_slice(row, 'train')
# for _, row in val_df.iterrows():
#     write_slice(row, 'val')



In [17]:
# Creates dataset to fit yolo format 

random.seed(42)

data_root    = config.TRAIN_DATASET_DIR
labels_csv   = config.FILTERED_TRAIN_LABELS 

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

box_size    = 32


for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id    = row["tomo_id"]
    z          = int(row["Motor axis 0"])
    y          = row["Motor axis 1"]
    x          = row["Motor axis 2"]
    img_width  = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name   = f"slice_{z:04d}.jpg"
    img_path   = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)

all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

# 1) Configuration for balancing
neg_pos_ratio = 1  # number of negative samples per positive

# 2) Separate positive & negative image paths
pos_images = list(positive_samples.keys())
neg_images = [img for img in all_images if img not in positive_samples]

# 3) Undersample negatives to at most neg_pos_ratio * #positives
num_pos = len(pos_images)
num_neg_keep = min(len(neg_images), num_pos * neg_pos_ratio)
neg_keep = random.sample(neg_images, num_neg_keep)

# 4) Combine & shuffle
balanced_images = pos_images + neg_keep
random.shuffle(balanced_images)

# 5) Train/val split on the balanced set
split_idx    = int(len(balanced_images) * 0.8)
train_images = balanced_images[:split_idx]
val_images   = balanced_images[split_idx:]

def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        # 1) Skip if the source image file doesn't exist
        if not os.path.isfile(img_path):
            print(f"⚠️  Skipping missing image: {img_path}")
            continue

        # 2) Reconstruct tomo_id and slice index as before
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z       = int(os.path.splitext(os.path.basename(img_path))[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        
        dst_img = os.path.join(img_out_dir,  f"{base_fn}.jpg")
        dst_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")

        # 3) Copy the image
        shutil.copy(img_path, dst_img)

        # 4) Write the label file (empty for negatives)
        lines = []
        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            cx = x / w
            cy = y / h
            bw = box_size * 2 / w
            bh = box_size * 2 / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        with open(dst_lbl, "w") as f:
            f.write("\n".join(lines))


# Process both sets
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images,   output_images_val,   output_labels_val)

print(f"Dataset created with {len(train_images)} training and {len(val_images)} validation images.") 


Dataset created with 32 training and 8 validation images.


In [5]:


# 1) Set paths – adjust these!
root_dir    = config.TRAIN_DATASET_DIR           # parent folder containing tomo_* subfolders
labels_csv  = config.TRAIN_LABELS_PATH      # original CSV with a 'tomo_id' column
output_csv  = config.FILTERED_TRAIN_LABELS  # where to write the filtered CSV

print(os.listdir(root_dir))
# 2) Detect all tomo_ids by listing subfolders
tomo_ids = [
    name for name in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, name))
]
print(f"Detected {len(tomo_ids)} tomo_id folders.")
print(tomo_ids)

# 3) Load your full labels CSV
df = pd.read_csv(labels_csv)

# 4) Filter rows where 'tomo_id' is in the detected folders
filtered_df = df[df['tomo_id'].isin(tomo_ids)]
print(f"Filtering yields {len(filtered_df)} rows out of {len(df)} total.")

# 5) Write out the new CSV
filtered_df.to_csv(output_csv, index=False)
print(f"Written filtered CSV to {output_csv}")

# 6) (Optional) Preview first few rows
print(filtered_df.head())


['tomo_3183d2', 'tomo_1af88d', 'tomo_0363f2', 'tomo_1446aa', 'tomo_8d231b', 'tomo_04d42b', 'tomo_6df2d6', 'tomo_56b9a3', 'tomo_3e7783', 'tomo_2483bb', 'tomo_0a8f05', 'tomo_49725c', 'tomo_1e9980', 'tomo_935ae0', 'tomo_79756f', 'tomo_30b580', 'tomo_072a16', 'tomo_6bc974', 'tomo_79a385', 'tomo_4469a7', 'tomo_9c0253', 'tomo_3c6038', 'tomo_88af60', 'tomo_62eea8', '.DS_Store', 'tomo_2e1f4c']
Detected 25 tomo_id folders.
['tomo_3183d2', 'tomo_1af88d', 'tomo_0363f2', 'tomo_1446aa', 'tomo_8d231b', 'tomo_04d42b', 'tomo_6df2d6', 'tomo_56b9a3', 'tomo_3e7783', 'tomo_2483bb', 'tomo_0a8f05', 'tomo_49725c', 'tomo_1e9980', 'tomo_935ae0', 'tomo_79756f', 'tomo_30b580', 'tomo_072a16', 'tomo_6bc974', 'tomo_79a385', 'tomo_4469a7', 'tomo_9c0253', 'tomo_3c6038', 'tomo_88af60', 'tomo_62eea8', 'tomo_2e1f4c']
Filtering yields 26 rows out of 737 total.
Written filtered CSV to /home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/raw/filtered_train_labels.csv
    row_id      tomo_id  Motor axis 0  Mo